# Backtracking (Tư duy Quay lui)

## 1. Định nghĩa

**Backtracking** là kỹ thuật "thử và sai" có hệ thống — xây dựng lời giải từng bước, kiểm tra ràng buộc sau mỗi bước, và **quay lui** khi đường đang đi không dẫn đến kết quả hợp lệ.

> *"If a candidate is no longer viable, the algorithm backtracks to try a different choice."*

### 4 pha hoạt động

```
1. BUILD    → xây dựng lời giải từng bước (thêm lựa chọn vào trạng thái)
2. CHECK    → kiểm tra ràng buộc: trạng thái hiện tại có hợp lệ không?
3. RECORD   → nếu đã hoàn chỉnh → lưu kết quả
4. UNDO     → xóa lựa chọn vừa thêm để thử nhánh khác (← đây là "backtrack")
```

## 2. Hình dung: Cây quyết định

Backtracking duyệt một **cây quyết định** theo DFS (depth-first). Mỗi nút là một trạng thái, mỗi nhánh là một lựa chọn.

```
                    []
                 /      \
           [1]              []
          /    \           /   \
       [1,2]  [1]       [2]    []
       /  \   / \       / \   / \
   [1,2,3][1,2][1,3][1] [2,3][2] [3] []
      ↑ lưu kết quả khi đến lá (index == n)
```

Khi gặp nhánh **vi phạm ràng buộc** → cắt tỉa sớm (**pruning**) → không đi tiếp xuống nhánh đó.

## 3. Template tổng quát

In [1]:
def backtrack(state, choices, results):
    # 1. BASE CASE: lời giải hoàn chỉnh → lưu lại
    if is_complete(state):
        results.append(state.copy())  # copy! không lưu tham chiếu
        return

    for choice in choices:
        # 2. PRUNING: bỏ qua lựa chọn không hợp lệ sớm
        if not is_valid(state, choice):
            continue

        # 3. APPLY: thêm lựa chọn vào trạng thái
        apply(state, choice)

        # 4. RECURSE: đệ quy với trạng thái mới
        backtrack(state, choices, results)

        # 5. UNDO: hoàn tác — đây là bước "backtrack" cốt lõi
        undo(state, choice)

> **Tại sao phải `.copy()` khi lưu?**  
> `state` là list dùng chung, nếu lưu trực tiếp thì các bước sau `.pop()` sẽ thay đổi luôn kết quả đã lưu.

## 4. Ví dụ 1: Tìm tất cả tập con (Subsets)

In [ ]:
def find_all_subsets(nums: list[int]) -> list[list[int]]:
    """Tìm tất cả tập con của nums.

    Args:
        nums: Danh sách số nguyên.

    Returns:
        Danh sách tất cả tập con (bao gồm tập rỗng).
    """
    results = []
    n = len(nums)

    def backtrack(index: int, current: list[int]) -> None:
        if index == n:                      # đã xét hết phần tử
            results.append(list(current))   # lưu bản sao
            return

        # Lựa chọn 1: KHÔNG lấy nums[index]
        backtrack(index + 1, current)

        # Lựa chọn 2: LẤY nums[index]
        current.append(nums[index])
        backtrack(index + 1, current)
        current.pop()  # UNDO

    backtrack(0, [])
    return results


# Test
print(find_all_subsets([1, 2, 3]))
# [[], [3], [2], [2,3], [1], [1,3], [1,2], [1,2,3]]

**Trace với `[1, 2, 3]`:**
```
bt(0,[])  → bt(1,[])  → bt(2,[])  → bt(3,[])   → lưu []
                                  → bt(3,[3])   → lưu [3]  | pop→[]
                      → bt(2,[2]) → bt(3,[2])   → lưu [2]
                                  → bt(3,[2,3]) → lưu [2,3]| pop→[2]| pop→[]
          → bt(1,[1]) → bt(2,[1]) → bt(3,[1])   → lưu [1]
                                  → bt(3,[1,3]) → lưu [1,3]| pop→[1]
                      → bt(2,[1,2])→bt(3,[1,2]) → lưu [1,2]
                                  → bt(3,[1,2,3])→lưu [1,2,3]
```
**Độ phức tạp:** O(2ⁿ) — mỗi phần tử có 2 lựa chọn.

## 5. Ví dụ 2: N-Queens

In [2]:
def solve_nqueens(n: int) -> list[list[str]]:
    """Đặt N quân hậu lên bàn cờ N×N sao cho không quân nào tấn công nhau.

    Args:
        n: Kích thước bàn cờ và số quân hậu.

    Returns:
        Danh sách tất cả cấu hình hợp lệ.
    """
    results = []
    cols_used = set()   # cột đã bị chiếm
    diag1_used = set()  # đường chéo \ : row - col = hằng số
    diag2_used = set()  # đường chéo / : row + col = hằng số
    board = [["." for _ in range(n)] for _ in range(n)]

    def backtrack(row: int) -> None:
        if row == n:  # đặt xong tất cả n hàng
            results.append([".".join(r) for r in board])
            return

        for col in range(n):
            # PRUNING: bỏ qua nếu cột hoặc đường chéo đã bị chiếm
            if col in cols_used or \
               (row - col) in diag1_used or \
               (row + col) in diag2_used:
                continue

            # APPLY
            cols_used.add(col)
            diag1_used.add(row - col)
            diag2_used.add(row + col)
            board[row][col] = "Q"

            backtrack(row + 1)

            # UNDO
            cols_used.remove(col)
            diag1_used.remove(row - col)
            diag2_used.remove(row + col)
            board[row][col] = "."

    backtrack(0)
    return results


# Test
solutions = solve_nqueens(4)
print(f"N=4: {len(solutions)} lời giải")
for s in solutions:
    print(s)

N=4: 2 lời giải
['..Q....', '......Q', 'Q......', '....Q..']
['....Q..', 'Q......', '......Q', '..Q....']


**Tại sao dùng `row - col` và `row + col` cho đường chéo?**
```
Bàn cờ 4×4:
       col: 0  1  2  3
row 0:      .  .  Q  .    row-col= 0-2=-2,  row+col= 0+2=2
row 1:      Q  .  .  .    row-col= 1-0=1,   row+col= 1+0=1
row 2:      .  .  .  Q    row-col= 2-3=-1,  row+col= 2+3=5
row 3:      .  Q  .  .    row-col= 3-1=2,   row+col= 3+1=4

Hai ô cùng đường chéo \ có cùng (row - col)
Hai ô cùng đường chéo / có cùng (row + col)
```
**Độ phức tạp:** O(n!) — mỗi hàng chọn 1 trong n cột, hàng sau loại bỏ dần.

## 6. Ví dụ 3: Giải mê cung (Maze Solver)

In [ ]:
def solve_maze(maze: list[list[str]]) -> bool:
    """Tìm đường đi từ 'S' đến 'E' trong mê cung.

    Args:
        maze: Lưới 2D gồm: 'S'=start, 'E'=end, '#'=tường, '.'=đường đi.

    Returns:
        True nếu tìm được đường đi.
    """
    rows, cols = len(maze), len(maze[0])
    grid = [list(row) for row in maze]  # copy để đánh dấu visited
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # lên, xuống, trái, phải

    def backtrack(r: int, c: int) -> bool:
        # BASE CASE: đến đích
        if grid[r][c] == 'E':
            return True

        # APPLY: đánh dấu đã thăm
        if grid[r][c] != 'S':
            grid[r][c] = 'V'

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            # PRUNING: trong biên, không phải tường, chưa thăm
            if 0 <= nr < rows and 0 <= nc < cols and \
               grid[nr][nc] != '#' and grid[nr][nc] != 'V':
                if backtrack(nr, nc):
                    return True

        # UNDO: bỏ đánh dấu (thử đường khác)
        if grid[r][c] != 'S':
            grid[r][c] = '.'
        return False

    # Tìm vị trí Start
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 'S':
                return backtrack(r, c)
    return False


# Test
maze = [
    ['S', '.', '#', '#'],
    ['#', '.', '.', '#'],
    ['#', '#', '.', '.'],
    ['#', '#', '#', 'E'],
]
print(solve_maze(maze))  # True — đường: S→(0,1)→(1,1)→(1,2)→(2,2)→(2,3)→E

**Trace bước đi:**
```
S . # #        S V # #        S V # #
# . . #   →   # V . #   →    # V V #   → ... → tìm thấy E
# # . .        # # . .        # # V .
# # # E        # # # E        # # # E

Nếu đường cụt → UNDO: đổi 'V' về '.' → thử hướng khác
```

## 7. Khi nào nên dùng Backtracking?

✅ Bài toán cần **tìm tất cả** hoặc **một** lời giải thỏa ràng buộc  
✅ Lời giải được **xây dựng từng bước** (incremental)  
✅ Tại mỗi bước có **tập lựa chọn rõ ràng**  
✅ Có thể **kiểm tra ràng buộc sớm** (pruning) để cắt tỉa nhánh vô ích  

| Bài toán | Ví dụ |
|---|---|
| Tổ hợp / Hoán vị | Subsets, Permutations, Combinations |
| Constraint satisfaction | N-Queens, Sudoku, Graph Coloring |
| Tìm đường | Maze, Word Search |
| Phân hoạch | Palindrome Partitioning |

## 8. Phân tích độ phức tạp

| Bài toán | Time | Space | Ghi chú |
|---|---|---|---|
| Subsets | O(2ⁿ) | O(n) | Mỗi phần tử: lấy hoặc không |
| Permutations | O(n!) | O(n) | n! hoán vị |
| N-Queens | O(n!) | O(n) | Pruning giúp giảm đáng kể |
| Maze | O(4ⁿ) worst | O(n) | n = số ô, 4 hướng mỗi ô |

> Backtracking thường có **độ phức tạp mũ** — nhưng **pruning** là vũ khí chính giúp giảm thời gian thực tế xuống đáng kể so với brute force.

## 9. So sánh Backtracking vs các kỹ thuật khác

| | Brute Force | Backtracking | Dynamic Programming |
|---|---|---|---|
| Cách tiếp cận | Thử tất cả | Thử + cắt tỉa sớm | Ghi nhớ kết quả con |
| Khi nào dùng | Bài nhỏ | Bài có ràng buộc | Bài có subproblem lặp lại |
| Time | Tệ nhất | Tốt hơn BF nhờ pruning | Thường tốt hơn |
| Tìm TẤT CẢ lời giải | ✅ | ✅ | ❌ (thường chỉ tìm optimal) |

## 10. Anti-patterns cần tránh

| Lỗi thường gặp | Cách tránh |
|---|---|
| Quên `.copy()` khi lưu kết quả | `results.append(list(current))` thay vì `results.append(current)` |
| Quên bước UNDO → kết quả sai | Mỗi `apply` phải có `undo` tương ứng |
| Không pruning → TLE | Kiểm tra ràng buộc **trước** khi đệ quy |
| Base case sai → đệ quy vô tận | Xác định rõ điều kiện dừng trước khi code |

## 11. Bài tập thực hành

| # | Bài toán | Kỹ thuật chính | Độ khó |
|---|---|---|---|
| 1 | Subsets | Lấy / không lấy | Easy |
| 2 | Permutations | Hoán vị có dùng `used[]` | Medium |
| 3 | Combination Sum | Có thể dùng lại phần tử | Medium |
| 4 | Word Search | Backtrack trên grid 2D | Medium |
| 5 | Palindrome Partitioning | Backtrack + kiểm tra palindrome | Medium |
| 6 | N-Queens | Pruning bằng set | Hard |
| 7 | Sudoku Solver | Constraint propagation | Hard |

## 12. Bài tập: Permutations

In [ ]:
def permutations(nums: list[int]) -> list[list[int]]:
    """Tìm tất cả hoán vị của nums.

    Args:
        nums: Danh sách số nguyên (không trùng lặp).

    Returns:
        Danh sách tất cả hoán vị.
    """
    results = []
    used = [False] * len(nums)  # đánh dấu phần tử đã dùng

    def backtrack(current: list[int]) -> None:
        if len(current) == len(nums):  # hoán vị đủ n phần tử
            results.append(list(current))
            return

        for i in range(len(nums)):
            if used[i]:
                continue  # PRUNING: bỏ qua phần tử đã dùng

            # APPLY
            used[i] = True
            current.append(nums[i])

            backtrack(current)

            # UNDO
            used[i] = False
            current.pop()

    backtrack([])
    return results


# Test
result = permutations([1, 2, 3])
print(f"Số hoán vị: {len(result)}")  # 3! = 6
for p in result:
    print(p)